# Validação de Fusão: openSMILE + Fingerprints — Classificação de Quadrantes Emocionais

Este notebook avalia empiricamente se as features de fingerprint acústico **complementam** as features openSMILE na classificação dos **quadrantes emocionais** (Q1–Q4 no espaço Valence × Arousal).

> **Nota metodológica:** o objetivo *não é provar previamente* que a fusão melhora, mas medir empiricamente se há ganho real, em quais cenários e em quais quadrantes.

**Cenários avaliados:**
1. openSMILE isolado
2. Fingerprint Rank isolado
3. Fingerprint Band isolado
4. Todos os Fingerprints isolados
5. openSMILE + Fingerprint Rank
6. openSMILE + Fingerprint Band
7. openSMILE + Todos os Fingerprints
8–10. Cenários com Raw Peaks (opcionais)

**Validação oficial:** `GroupKFold(n_splits=5)` agrupado por `song_id` — sem vazamento entre músicas.  
**Métrica principal:** Macro F1, seguida de Balanced Accuracy.

## 1. Imports e configuração

In [1]:
import os
import warnings
import re
import json
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional, Tuple

os.environ['LOKY_MAX_CPU_COUNT'] = str(os.cpu_count() or 1)
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from IPython.display import display, Markdown
from tqdm.auto import tqdm
from scipy.stats import pearsonr

from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.ensemble import (
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

try:
    from sklearn.ensemble import HistGradientBoostingClassifier
    HAS_HGBC = True
except ImportError:
    HAS_HGBC = False

try:
    from sklearn.svm import SVC
    HAS_SVM = True
except ImportError:
    HAS_SVM = False

print(f'pandas {pd.__version__} | numpy {np.__version__}')
print(f'HistGradientBoostingClassifier: {HAS_HGBC} | SVC: {HAS_SVM}')
print(f'LOKY_MAX_CPU_COUNT: {os.environ["LOKY_MAX_CPU_COUNT"]}')

pandas 2.1.4 | numpy 1.26.4
HistGradientBoostingClassifier: True | SVC: True
LOKY_MAX_CPU_COUNT: 12


In [2]:
@dataclass
class Config:
    project_dir: Path = Path(r'C:\dev\python\TCC Ajustado')
    random_state: int = 42
    n_splits: int = 5
    n_estimators: int = 200
    n_jobs: int = -1
    use_raw_peaks: bool = True
    run_svm: bool = False
    export_html: bool = True
    show_figures: bool = True

    QUADRANT_ORDER: List[str] = field(default_factory=lambda: [
        'Q1_High_Valence_High_Arousal',
        'Q2_Low_Valence_High_Arousal',
        'Q3_Low_Valence_Low_Arousal',
        'Q4_High_Valence_Low_Arousal',
    ])
    QUADRANT_SHORT: Dict[str, str] = field(default_factory=lambda: {
        'Q1_High_Valence_High_Arousal': 'Q1',
        'Q2_Low_Valence_High_Arousal':  'Q2',
        'Q3_Low_Valence_Low_Arousal':   'Q3',
        'Q4_High_Valence_Low_Arousal':  'Q4',
    })

    META_COLS: List[str] = field(default_factory=lambda: [
        'song_id', 'filename', 'block_id', 'block_start_sec', 'block_end_sec',
        'block_duration_sec', 'valence', 'arousal', 'quadrant', 'quadrant_label',
        'quadrant_label_std', 'method', 'title', 'artist', 'genre',
        'band_name', 'band_id', 'band_low_hz', 'band_high_hz', 'topk_per_band',
    ])

    def data_dir(self) -> Path:
        return self.project_dir / 'Dados'

    def output_dir(self) -> Path:
        p = self.project_dir / 'Dados' / 'pycaret_fusion_quadrants_validation'
        p.mkdir(parents=True, exist_ok=True)
        return p

    def figures_dir(self) -> Path:
        p = self.output_dir() / 'figures'
        p.mkdir(parents=True, exist_ok=True)
        return p


cfg = Config()
QUADRANT_COL = 'quadrant_label'
JOIN_KEYS    = ['song_id', 'block_start_sec', 'block_end_sec']
print('Output dir:', cfg.output_dir())
print('Figures dir:', cfg.figures_dir())

Output dir: C:\dev\python\TCC Ajustado\Dados\pycaret_fusion_quadrants_validation
Figures dir: C:\dev\python\TCC Ajustado\Dados\pycaret_fusion_quadrants_validation\figures


## 2. Funções utilitárias

In [3]:
def numeric_valid_cols(df: pd.DataFrame, exclude: List[str]) -> List[str]:
    excl = set(exclude)
    out = []
    for col in df.columns:
        if col in excl:
            continue
        s = pd.to_numeric(df[col], errors='coerce')
        if s.notna().sum() > 0 and s.nunique(dropna=True) > 1:
            out.append(col)
    return out


def round_times(df: pd.DataFrame, decimals: int = 1) -> pd.DataFrame:
    for col in ['block_start_sec', 'block_end_sec']:
        if col in df.columns:
            df[col] = df[col].round(decimals)
    return df


def add_prefix(df: pd.DataFrame, prefix: str, exclude: List[str]) -> Tuple[pd.DataFrame, List[str]]:
    excl = set(exclude)
    rename = {c: f'{prefix}{c}' for c in df.columns if c not in excl and not c.startswith(prefix)}
    df = df.rename(columns=rename)
    return df, [rename.get(c, c) for c in rename]


def ensure_quadrant_label(df: pd.DataFrame, v_thresh: float = 0.0, a_thresh: float = 0.0) -> pd.DataFrame:
    if QUADRANT_COL in df.columns and df[QUADRANT_COL].notna().any():
        return df
    if 'valence' not in df.columns or 'arousal' not in df.columns:
        raise ValueError('Sem quadrant_label nem valence/arousal para derivá-lo.')
    v = pd.to_numeric(df['valence'], errors='coerce')
    a = pd.to_numeric(df['arousal'], errors='coerce')
    conditions = [
        (v >= v_thresh) & (a >= a_thresh),
        (v <  v_thresh) & (a >= a_thresh),
        (v <  v_thresh) & (a <  a_thresh),
        (v >= v_thresh) & (a <  a_thresh),
    ]
    choices = cfg.QUADRANT_ORDER
    df = df.copy()
    df[QUADRANT_COL] = np.select(conditions, choices, default=pd.NA)
    print(f'[INFO] quadrant_label derivado: {df[QUADRANT_COL].value_counts().to_dict()}')
    return df


def feature_origin(feat: str) -> str:
    if feat.startswith('opensmile__'):
        return 'openSMILE'
    if feat.startswith('fprank__'):
        return 'Fingerprint Rank'
    if feat.startswith('fpband__'):
        return 'Fingerprint Band'
    if feat.startswith('rawpeaks_rank__'):
        return 'Raw Peaks Rank'
    if feat.startswith('rawpeaks_band__'):
        return 'Raw Peaks Band'
    return 'outros'


ORIGIN_COLORS = {
    'openSMILE':         '#2196F3',
    'Fingerprint Rank':  '#FF9800',
    'Fingerprint Band':  '#4CAF50',
    'Raw Peaks Rank':    '#9C27B0',
    'Raw Peaks Band':    '#E91E63',
    'outros':            '#9E9E9E',
}

SCENARIO_LABELS = {
    '1_opensmile_only':      '1. openSMILE',
    '2_fprank_only':         '2. FP Rank',
    '3_fpband_only':         '3. FP Band',
    '4_allfp_only':          '4. All FP',
    '5_os_fprank':           '5. OS+FPRank',
    '6_os_fpband':           '6. OS+FPBand',
    '7_os_allfp':            '7. OS+AllFP',
    '8_rawpeaks_only':       '8. RawPks',
    '9_os_rawpeaks':         '9. OS+RawPks',
    '10_os_allfp_rawpeaks':  '10. OS+AllFP+Raw',
}

def slabel(s): return SCENARIO_LABELS.get(s, s)

def prefer_norm_magnitude(feat_list: List[str], df: pd.DataFrame) -> Tuple[List[str], List[str]]:
    """
    Remove features de magnitude em dB quando existe versão normalizada equivalente.

    Padrões substituídos (col_db → col_norm, se col_norm estiver no DataFrame):
      *_magnitude_db        → *_magnitude_norm_block    (fpband/fprank top-k por bloco)
      *_mag_*_db            → *_mag_*_norm_block        (fpband média/std por bloco)
      *_energy_db_<banda>   → *_energy_norm_<banda>     (fpband energia por banda)
      *_magnitude_db_<stat> → *_magnitude_norm_<stat>   (rawpeaks_band)
      *_magnitude_mean      → *_magnitude_mean_norm_block  (fprank média global)
    """
    df_cols = set(df.columns)
    keep: List[str] = []
    dropped: List[str] = []

    for col in feat_list:
        norm_col = None

        if col.endswith('_magnitude_db'):
            norm_col = col[: -len('_magnitude_db')] + '_magnitude_norm_block'
        elif col.endswith('_db') and '_mag_' in col and '_norm' not in col:
            norm_col = col[: -len('_db')] + '_norm_block'
        elif 'energy_db_' in col:
            norm_col = col.replace('energy_db_', 'energy_norm_')
        elif '_magnitude_db_' in col:
            norm_col = col.replace('_magnitude_db_', '_magnitude_norm_')
        elif '_norm_' not in col and col.endswith('_magnitude_mean'):
            norm_col = col + '_norm_block'

        if norm_col and norm_col in df_cols:
            dropped.append(col)
        else:
            keep.append(col)

    return keep, dropped

## 3. Carregamento dos dados

In [4]:
data_dir = cfg.data_dir()

# ── openSMILE ──────────────────────────────────────────────────────────────
os_df = pd.read_parquet(data_dir / 'comparative_outputs' / 'common' / 'opensmile_blocks.parquet')
os_df = round_times(os_df)
os_df = ensure_quadrant_label(os_df)

os_raw_feat = [c for c in os_df.columns if c.startswith('os_')]
os_rename   = {c: c.replace('os_mean__', 'opensmile__').replace('os_std__', 'opensmile__std__').replace('os_', 'opensmile__') for c in os_raw_feat}
os_df = os_df.rename(columns=os_rename)
os_feat = list(os_rename.values())

print(f'openSMILE: {os_df.shape[0]} blocos | {len(os_feat)} features | {os_df["song_id"].nunique()} músicas')
print('Distribuição dos quadrantes:')
print(os_df[QUADRANT_COL].value_counts().reindex(cfg.QUADRANT_ORDER).to_string())

openSMILE: 6506 blocos | 520 features | 1802 músicas
Distribuição dos quadrantes:
quadrant_label
Q1_High_Valence_High_Arousal    3217
Q2_Low_Valence_High_Arousal     1019
Q3_Low_Valence_Low_Arousal      1382
Q4_High_Valence_Low_Arousal      888


In [5]:
# ── Fingerprint Rank (bloco) ───────────────────────────────────────────────
fprank_raw = pd.read_parquet(data_dir / 'fingerprint_rank' / 'fingerprint_rank.parquet')
fprank_raw = round_times(fprank_raw)
fprank_raw, _ = add_prefix(fprank_raw, 'fprank__', cfg.META_COLS)
fprank_feat = numeric_valid_cols(fprank_raw, exclude=cfg.META_COLS)
print(f'FP Rank:  {fprank_raw.shape[0]} blocos | {len(fprank_feat)} features')

# ── Fingerprint Band (bloco) ───────────────────────────────────────────────
fpband_raw = pd.read_parquet(data_dir / 'fingerprint_band_rank' / 'fingerprint_band_rank.parquet')
fpband_raw = round_times(fpband_raw)
fpband_raw, _ = add_prefix(fpband_raw, 'fpband__', cfg.META_COLS)
fpband_feat = numeric_valid_cols(fpband_raw, exclude=cfg.META_COLS)
print(f'FP Band:  {fpband_raw.shape[0]} blocos | {len(fpband_feat)} features')

FP Rank:  6506 blocos | 207 features
FP Band:  6506 blocos | 302 features


## 4. Agregação dos Raw Peaks (longo → bloco)

In [6]:
def aggregate_raw_peaks(path: Path, prefix: str, by_band: bool = False) -> Tuple[Optional[pd.DataFrame], List[str]]:
    """Agrega raw peaks (formato longo) para nível de bloco com estatísticas descritivas."""
    if not path.exists():
        print(f'  [SKIP] {path.name} não encontrado')
        return None, []
    raw = pd.read_parquet(path)
    raw = round_times(raw)
    keys = [k for k in ['song_id', 'block_id', 'block_start_sec', 'block_end_sec'] if k in raw.columns]
    num_candidates = ['frequency_hz', 'magnitude_db', 'magnitude_norm', 'magnitude',
                      'midi_note', 'octave', 'semitone', 'peak_rank_global']
    num_cols = [c for c in num_candidates if c in raw.columns]
    for c in num_cols:
        raw[c] = pd.to_numeric(raw[c], errors='coerce')

    def _agg_block(sub_df, pref):
        g = sub_df.groupby(keys)
        agg = g[num_cols].agg(['mean', 'std', 'max'])
        agg.columns = [f'{pref}{col}_{stat}' for col, stat in agg.columns]
        agg[f'{pref}n_peaks'] = g.size()
        return agg.reset_index()

    if by_band and 'band_name' in raw.columns:
        frames = []
        for band in sorted(raw['band_name'].dropna().unique()):
            sub = raw[raw['band_name'] == band]
            frames.append(_agg_block(sub, f'{prefix}{band}_'))
        if not frames:
            return None, []
        merged = frames[0]
        for pf in frames[1:]:
            merged = merged.merge(pf, on=keys, how='outer')
    else:
        merged = _agg_block(raw, prefix)

    feat_cols = [c for c in merged.columns if c.startswith(prefix)]
    print(f'  {prefix}: {merged.shape[0]} blocos | {len(feat_cols)} features')
    return merged, feat_cols


In [7]:
print('Agregando raw peaks...')

raw_rank_df, raw_rank_feat = aggregate_raw_peaks(
    data_dir / 'fingerprint_rank' / 'fingerprint_rank_raw_peaks.parquet',
    prefix='rawpeaks_rank__', by_band=False
)
raw_band_df, raw_band_feat = aggregate_raw_peaks(
    data_dir / 'fingerprint_band_rank' / 'fingerprint_band_rank_raw_peaks.parquet',
    prefix='rawpeaks_band__', by_band=True
)

HAS_RAW_RANK = cfg.use_raw_peaks and raw_rank_df is not None
HAS_RAW_BAND = cfg.use_raw_peaks and raw_band_df is not None
HAS_RAW = HAS_RAW_RANK or HAS_RAW_BAND

all_raw_feat = []
if HAS_RAW_RANK: all_raw_feat.extend(raw_rank_feat)
if HAS_RAW_BAND: all_raw_feat.extend(raw_band_feat)
all_raw_feat = list(dict.fromkeys(all_raw_feat))

print(f'Raw peaks disponíveis: {HAS_RAW} (rank={HAS_RAW_RANK}, band={HAS_RAW_BAND})')

Agregando raw peaks...
  rawpeaks_rank__: 6506 blocos | 19 features
  rawpeaks_band__: 6506 blocos | 100 features
Raw peaks disponíveis: True (rank=True, band=True)


## 5. Montagem do dataset integrado

In [8]:
def left_join(base: pd.DataFrame, right: pd.DataFrame, feat_cols: List[str], name: str) -> Tuple[pd.DataFrame, List[str]]:
    keys_present = [k for k in JOIN_KEYS if k in right.columns]
    right_sel = right[keys_present + feat_cols].drop_duplicates(subset=keys_present)
    merged = base.merge(right_sel, on=keys_present, how='left')
    added  = [f for f in feat_cols if f in merged.columns]
    cov    = merged[added[0]].notna().mean() if added else 0.0
    print(f'  [{name}] {merged.shape[0]} blocos | {len(added)} features | coverage {cov:.1%}')
    return merged, added


# base: openSMILE com metadados
base_meta_cols = [c for c in [
    'song_id', 'filename', 'block_id', 'block_start_sec', 'block_end_sec',
    'block_duration_sec', 'valence', 'arousal', 'quadrant', QUADRANT_COL
] if c in os_df.columns]

dataset = os_df[base_meta_cols + os_feat].copy()

dataset, fprank_feat   = left_join(dataset, fprank_raw, fprank_feat,   'FP Rank')
dataset, fpband_feat   = left_join(dataset, fpband_raw, fpband_feat,   'FP Band')

if HAS_RAW_RANK:
    dataset, raw_rank_feat = left_join(dataset, raw_rank_df, raw_rank_feat, 'Raw Rank')
if HAS_RAW_BAND:
    dataset, raw_band_feat = left_join(dataset, raw_band_df, raw_band_feat, 'Raw Band')

all_raw_feat = list(dict.fromkeys(
    (raw_rank_feat if HAS_RAW_RANK else []) +
    (raw_band_feat if HAS_RAW_BAND else [])
))

print(f'\nDataset integrado: {dataset.shape}')
print(f'Músicas: {dataset["song_id"].nunique()}')

  [FP Rank] 6506 blocos | 207 features | coverage 100.0%
  [FP Band] 6506 blocos | 302 features | coverage 100.0%
  [Raw Rank] 6506 blocos | 19 features | coverage 99.5%
  [Raw Band] 6506 blocos | 100 features | coverage 100.0%

Dataset integrado: (6506, 1158)
Músicas: 1802


In [9]:
# refinar: garantir que features são numéricas e não-constantes no dataset final
all_meta = set(cfg.META_COLS + ['block_id', 'block_duration_sec', 'filename', 'quadrant'])

def vfeats(cols):
    return [c for c in numeric_valid_cols(dataset, exclude=list(all_meta)) if c in cols]

os_feat_v      = vfeats(os_feat)
fprank_feat_v  = vfeats(fprank_feat)
fpband_feat_v  = vfeats(fpband_feat)
rawrank_feat_v = vfeats(raw_rank_feat) if HAS_RAW_RANK else []
rawband_feat_v = vfeats(raw_band_feat) if HAS_RAW_BAND else []

# ── Priorizar magnitude normalizada: remover versões em dB com par normalizado ──
fprank_feat_v,  fprank_dropped  = prefer_norm_magnitude(fprank_feat_v,  dataset)
fpband_feat_v,  fpband_dropped  = prefer_norm_magnitude(fpband_feat_v,  dataset)
rawrank_feat_v, rawrank_dropped = prefer_norm_magnitude(rawrank_feat_v, dataset)
rawband_feat_v, rawband_dropped = prefer_norm_magnitude(rawband_feat_v, dataset)

_all_dropped = fprank_dropped + fpband_dropped + rawrank_dropped + rawband_dropped
print(f'Features removidas (magnitude dB → normalizada disponível): {len(_all_dropped)}')
for _d in sorted(_all_dropped)[:30]:
    print(f'  - {_d}')
if len(_all_dropped) > 30:
    print(f'  ... e mais {len(_all_dropped) - 30}')
print()

all_fp_feat    = list(dict.fromkeys(fprank_feat_v + fpband_feat_v))
all_raw_feat_v = list(dict.fromkeys(rawrank_feat_v + rawband_feat_v))

print(f'openSMILE features:     {len(os_feat_v)}')
print(f'FP Rank features:       {len(fprank_feat_v)}')
print(f'FP Band features:       {len(fpband_feat_v)}')
print(f'Raw Peaks features:     {len(all_raw_feat_v)}')

Features removidas (magnitude dB → normalizada disponível): 97
  - fpband__energy_db_high
  - fpband__energy_db_low
  - fpband__energy_db_mid
  - fpband__energy_db_very_high
  - fpband__fp_high_mag_mean_db
  - fpband__fp_high_mag_std_db
  - fpband__fp_high_top_10_magnitude_db
  - fpband__fp_high_top_1_magnitude_db
  - fpband__fp_high_top_2_magnitude_db
  - fpband__fp_high_top_3_magnitude_db
  - fpband__fp_high_top_4_magnitude_db
  - fpband__fp_high_top_5_magnitude_db
  - fpband__fp_high_top_6_magnitude_db
  - fpband__fp_high_top_7_magnitude_db
  - fpband__fp_high_top_8_magnitude_db
  - fpband__fp_high_top_9_magnitude_db
  - fpband__fp_low_mag_mean_db
  - fpband__fp_low_mag_std_db
  - fpband__fp_low_top_10_magnitude_db
  - fpband__fp_low_top_1_magnitude_db
  - fpband__fp_low_top_2_magnitude_db
  - fpband__fp_low_top_3_magnitude_db
  - fpband__fp_low_top_4_magnitude_db
  - fpband__fp_low_top_5_magnitude_db
  - fpband__fp_low_top_6_magnitude_db
  - fpband__fp_low_top_7_magnitude_db
  - fp

In [10]:
# definição dos cenários
SCENARIOS: Dict[str, List[str]] = {
    '1_opensmile_only': os_feat_v,
    '2_fprank_only':    fprank_feat_v,
    '3_fpband_only':    fpband_feat_v,
    '4_allfp_only':     all_fp_feat,
    '5_os_fprank':      list(dict.fromkeys(os_feat_v + fprank_feat_v)),
    '6_os_fpband':      list(dict.fromkeys(os_feat_v + fpband_feat_v)),
    '7_os_allfp':       list(dict.fromkeys(os_feat_v + all_fp_feat)),
}

if HAS_RAW and all_raw_feat_v:
    SCENARIOS['8_rawpeaks_only']      = all_raw_feat_v
    SCENARIOS['9_os_rawpeaks']        = list(dict.fromkeys(os_feat_v + all_raw_feat_v))
    SCENARIOS['10_os_allfp_rawpeaks'] = list(dict.fromkeys(os_feat_v + all_fp_feat + all_raw_feat_v))

for name, feats in SCENARIOS.items():
    print(f'  {name}: {len(feats)} features')

  1_opensmile_only: 520 features
  2_fprank_only: 176 features
  3_fpband_only: 248 features
  4_allfp_only: 424 features
  5_os_fprank: 696 features
  6_os_fpband: 768 features
  7_os_allfp: 944 features
  8_rawpeaks_only: 98 features
  9_os_rawpeaks: 618 features
  10_os_allfp_rawpeaks: 1042 features


## 6. Modelos e avaliação GroupKFold

In [11]:
def build_models(cfg: Config) -> Dict[str, object]:
    models = {
        'RidgeClassifier':       RidgeClassifier(class_weight='balanced', random_state=cfg.random_state),
        'LogisticRegression':    LogisticRegression(max_iter=3000, class_weight='balanced', random_state=cfg.random_state, n_jobs=cfg.n_jobs),
        'RandomForest':          RandomForestClassifier(n_estimators=cfg.n_estimators, class_weight='balanced_subsample', random_state=cfg.random_state, n_jobs=cfg.n_jobs),
        'ExtraTrees':            ExtraTreesClassifier(n_estimators=cfg.n_estimators, class_weight='balanced', random_state=cfg.random_state, n_jobs=cfg.n_jobs),
        # GradientBoostingClassifier removido: sequencial e muito lento com n_estimators=200.
        # HistGradientBoostingClassifier abaixo é o substituto moderno (paralelizável, nativo para NaN).
    }
    if HAS_HGBC:
        models['HistGradientBoosting'] = HistGradientBoostingClassifier(random_state=cfg.random_state)
    if HAS_SVM and cfg.run_svm:
        models['SVM_rbf'] = SVC(kernel='rbf', class_weight='balanced', random_state=cfg.random_state, probability=False)
    return models


def build_pipeline(model, scale: bool = True) -> Pipeline:
    steps = [('imputer', SimpleImputer(strategy='median'))]
    if scale:
        steps.append(('scaler', StandardScaler()))
    steps.append(('model', model))
    return Pipeline(steps)


def classification_metrics(y_true, y_pred, labels) -> Dict[str, float]:
    return {
        'accuracy':          accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'macro_f1':          f1_score(y_true, y_pred, average='macro', zero_division=0),
        'weighted_f1':       f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'macro_precision':   precision_score(y_true, y_pred, average='macro', zero_division=0),
        'macro_recall':      recall_score(y_true, y_pred, average='macro', zero_division=0),
    }


def per_class_metrics(y_true, y_pred, labels) -> Dict[str, float]:
    result = {}
    for label in labels:
        mask_t = np.array(y_true) == label
        mask_p = np.array(y_pred) == label
        tp = (mask_t & mask_p).sum()
        fp = (~mask_t & mask_p).sum()
        fn = (mask_t & ~mask_p).sum()
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
        short = cfg.QUADRANT_SHORT.get(label, label)
        result[f'{short}_precision'] = prec
        result[f'{short}_recall']    = rec
        result[f'{short}_f1']        = f1
        result[f'{short}_support']   = mask_t.sum()
    return result


def get_importance(pipeline, feat_names) -> Optional[np.ndarray]:
    est = pipeline.named_steps.get('model')
    if hasattr(est, 'feature_importances_'):
        return est.feature_importances_
    if hasattr(est, 'coef_'):
        coef = est.coef_
        return np.abs(coef).mean(axis=0) if coef.ndim > 1 else np.abs(coef)
    return None

In [12]:
gkf    = GroupKFold(n_splits=cfg.n_splits)
models = build_models(cfg)

NEEDS_SCALE = {'RidgeClassifier', 'LogisticRegression', 'SVM_rbf'}

y_all    = dataset[QUADRANT_COL].astype(str)
groups   = dataset['song_id'].astype(str).values
labels   = [q for q in cfg.QUADRANT_ORDER if q in set(y_all)]

print(f'Classes: {labels}')
print(f'Músicas (grupos): {len(set(groups))} | Folds: {cfg.n_splits}')

fold_records       = []
importance_records = []
cm_records         = []   # confusion matrices acumuladas (OOF)

# OOF predictions por cenário e modelo (para confusion matrix)
oof_store: Dict[str, Dict[str, np.ndarray]] = {}

total_iter = sum(len(models) for _ in SCENARIOS)
pbar = tqdm(total=total_iter, desc='Experimentos')

for scenario_name, feat_cols in SCENARIOS.items():
    feat_cols = [f for f in feat_cols if f in dataset.columns]
    if not feat_cols:
        print(f'[WARN] {scenario_name}: nenhuma feature válida — pulando')
        pbar.update(len(models))
        continue

    X_all = dataset[feat_cols].apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan)
    valid_mask = y_all.notna() & X_all.notna().any(axis=1)
    X_v = X_all[valid_mask].values
    y_v = y_all[valid_mask].values
    g_v = groups[valid_mask]

    oof_store[scenario_name] = {}

    for model_name, base_model in models.items():
        scale   = model_name in NEEDS_SCALE
        oof_arr = np.full(len(y_v), fill_value='', dtype=object)
        fold_imps = []

        for fold_idx, (tr_idx, va_idx) in enumerate(gkf.split(X_v, y_v, g_v)):
            pipe = build_pipeline(clone(base_model), scale=scale)
            X_tr, X_va = X_v[tr_idx], X_v[va_idx]
            y_tr, y_va = y_v[tr_idx], y_v[va_idx]
            try:
                pipe.fit(X_tr, y_tr)
                pred = pipe.predict(X_va)
                oof_arr[va_idx] = pred
                mets  = classification_metrics(y_va, pred, labels)
                pmets = per_class_metrics(y_va, pred, labels)
                status = 'ok'
                imp = get_importance(pipe, feat_cols)
                if imp is not None and len(imp) == len(feat_cols):
                    fold_imps.append(imp)
            except Exception as e:
                mets   = {k: np.nan for k in ['accuracy','balanced_accuracy','macro_f1','weighted_f1','macro_precision','macro_recall']}
                pmets  = {}
                status = f'error:{e}'

            fold_records.append({
                'scenario': scenario_name, 'model': model_name, 'fold': fold_idx,
                'n_features': len(feat_cols), 'n_train': len(tr_idx), 'n_val': len(va_idx),
                'status': status, **mets, **pmets,
            })

        oof_store[scenario_name][model_name] = oof_arr

        if fold_imps:
            mean_imp = np.mean(fold_imps, axis=0)
            for feat, val in zip(feat_cols, mean_imp):
                importance_records.append({
                    'scenario': scenario_name, 'model': model_name,
                    'feature': feat, 'importance': val,
                })

        pbar.update(1)

pbar.close()
fold_df = pd.DataFrame(fold_records)
print(f'\nRegistros de folds: {len(fold_df)}')

Classes: ['Q1_High_Valence_High_Arousal', 'Q2_Low_Valence_High_Arousal', 'Q3_Low_Valence_Low_Arousal', 'Q4_High_Valence_Low_Arousal']
Músicas (grupos): 1802 | Folds: 5


Experimentos:   0%|          | 0/50 [00:00<?, ?it/s]

  File "c:\Users\felip\miniconda3\envs\pycaret311\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\felip\miniconda3\envs\pycaret311\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\felip\miniconda3\envs\pycaret311\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\Users\felip\miniconda3\envs\pycaret311\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^



Registros de folds: 250


## 7. Agregação dos resultados

In [13]:
ok_df = fold_df[fold_df['status'] == 'ok'].copy()

metric_cols = ['accuracy', 'balanced_accuracy', 'macro_f1', 'weighted_f1', 'macro_precision', 'macro_recall']

agg_dict_base = {}
for m in metric_cols:
    if m in ok_df.columns:
        agg_dict_base[f'{m}_mean'] = (m, 'mean')
        agg_dict_base[f'{m}_std']  = (m, 'std')

# per-class F1 mean
per_class_f1_cols = [c for c in ok_df.columns if c.endswith('_f1') and c not in metric_cols]
for pc in per_class_f1_cols:
    agg_dict_base[f'{pc}_mean'] = (pc, 'mean')

agg_dict_base['n_folds'] = ('fold', 'count')

agg_df = (
    ok_df.groupby(['scenario', 'model', 'n_features'])
    .agg(**agg_dict_base)
    .reset_index()
    .sort_values(['scenario', 'macro_f1_mean'], ascending=[True, False])
)

display(agg_df[['scenario','model','n_features','macro_f1_mean','macro_f1_std','balanced_accuracy_mean','accuracy_mean']].head(30))

,scenario,model,n_features,macro_f1_mean,macro_f1_std,balanced_accuracy_mean,accuracy_mean
1,10_os_allfp_rawpeaks,HistGradientBoosting,1042,0.482592,0.018243,0.481548,0.628803
4,10_os_allfp_rawpeaks,RidgeClassifier,1042,0.459693,0.029585,0.471244,0.522443
2,10_os_allfp_rawpeaks,LogisticRegression,1042,0.452964,0.023907,0.463018,0.519214
3,10_os_allfp_rawpeaks,RandomForest,1042,0.387108,0.008735,0.412241,0.602053
0,10_os_allfp_rawpeaks,ExtraTrees,1042,0.371396,0.017007,0.399431,0.595139
7,1_opensmile_only,LogisticRegression,520,0.468017,0.032614,0.484306,0.520300
9,1_opensmile_only,RidgeClassifier,520,0.467883,0.025560,0.484393,0.520758
6,1_opensmile_only,HistGradientBoosting,520,0.465161,0.023377,0.465464,0.614978
8,1_opensmile_only,RandomForest,520,0.395478,0.011605,0.414337,0.599599
5,1_opensmile_only,ExtraTrees,520,0.376082,0.009183,0.401067,0.593449


In [14]:
# melhor modelo por cenário
best_by_scenario = (
    agg_df.sort_values('macro_f1_mean', ascending=False)
    .groupby('scenario')
    .first()
    .reset_index()
    .rename(columns={'model': 'best_model'})
)

best_overall = best_by_scenario.sort_values('macro_f1_mean', ascending=False).iloc[0]
print('Melhor cenário geral:')
print(f"  Cenário: {best_overall['scenario']}")
print(f"  Modelo:  {best_overall['best_model']}")
print(f"  Macro F1:  {best_overall['macro_f1_mean']:.4f} ± {best_overall.get('macro_f1_std', 0):.4f}")
print(f"  Balanced Acc: {best_overall['balanced_accuracy_mean']:.4f}")

display(Markdown('### Melhor modelo por cenário'))
display(best_by_scenario[['scenario','best_model','macro_f1_mean','macro_f1_std','balanced_accuracy_mean','accuracy_mean']].sort_values('scenario'))

Melhor cenário geral:
  Cenário: 10_os_allfp_rawpeaks
  Modelo:  HistGradientBoosting
  Macro F1:  0.4826 ± 0.0182
  Balanced Acc: 0.4815


### Melhor modelo por cenário

,scenario,best_model,macro_f1_mean,macro_f1_std,balanced_accuracy_mean,accuracy_mean
0,10_os_allfp_rawpeaks,HistGradientBoosting,0.482592,0.018243,0.481548,0.628803
1,1_opensmile_only,LogisticRegression,0.468017,0.032614,0.484306,0.520300
2,2_fprank_only,LogisticRegression,0.431237,0.013825,0.445173,0.489397
3,3_fpband_only,LogisticRegression,0.404186,0.019248,0.424547,0.443896
4,4_allfp_only,LogisticRegression,0.441557,0.023476,0.458988,0.490781
5,5_os_fprank,HistGradientBoosting,0.475220,0.016015,0.474997,0.622657
6,6_os_fpband,HistGradientBoosting,0.478336,0.020226,0.475535,0.620659
7,7_os_allfp,HistGradientBoosting,0.481370,0.027871,0.478903,0.624809
8,8_rawpeaks_only,LogisticRegression,0.407382,0.016996,0.427278,0.452040
9,9_os_rawpeaks,HistGradientBoosting,0.474485,0.021090,0.473122,0.621736


## 8. Comparação: fusão vs openSMILE isolado

In [15]:
def classify_gain_clf(row) -> str:
    d_f1  = row.get('delta_macro_f1', 0.0)
    d_ba  = row.get('delta_balanced_accuracy', 0.0)
    if d_f1 > 0.015 and d_ba > 0.01:
        return 'melhora consistente'
    elif d_f1 > 0.01:
        return 'melhora em Macro F1'
    elif d_ba > 0.01:
        return 'melhora em Balanced Accuracy'
    elif d_f1 > 0.003 or d_ba > 0.003:
        return 'melhora pequena'
    return 'sem melhora relevante'


os_ref = best_by_scenario[best_by_scenario['scenario'] == '1_opensmile_only'][[
    'best_model', 'macro_f1_mean', 'balanced_accuracy_mean', 'accuracy_mean', 'weighted_f1_mean'
]].rename(columns={
    'best_model':              'os_best_model',
    'macro_f1_mean':           'os_macro_f1',
    'balanced_accuracy_mean':  'os_balanced_accuracy',
    'accuracy_mean':           'os_accuracy',
    'weighted_f1_mean':        'os_weighted_f1',
})

if not os_ref.empty:
    os_mf1 = float(os_ref['os_macro_f1'].values[0])
    os_ba  = float(os_ref['os_balanced_accuracy'].values[0])
    os_acc = float(os_ref['os_accuracy'].values[0])
    os_wf1 = float(os_ref.get('os_weighted_f1', pd.Series([np.nan])).values[0])
    print(f'openSMILE isolado — Macro F1: {os_mf1:.4f} | Balanced Acc: {os_ba:.4f} | Accuracy: {os_acc:.4f}')
else:
    os_mf1 = os_ba = os_acc = os_wf1 = np.nan
    print('[WARN] Sem resultados para openSMILE isolado')

fusion_scenarios = [s for s in best_by_scenario['scenario'] if s != '1_opensmile_only']
delta_df = best_by_scenario[best_by_scenario['scenario'].isin(fusion_scenarios)].copy()

delta_df['delta_macro_f1']          = delta_df['macro_f1_mean']          - os_mf1
delta_df['delta_balanced_accuracy'] = delta_df['balanced_accuracy_mean'] - os_ba
delta_df['delta_accuracy']          = delta_df['accuracy_mean']          - os_acc
delta_df['delta_macro_f1_pct']      = 100 * delta_df['delta_macro_f1'] / (os_mf1 or 1e-9)
delta_df['ganho']                   = delta_df.apply(classify_gain_clf, axis=1)

display(Markdown('### Delta vs openSMILE isolado'))
display(delta_df[['scenario','best_model','macro_f1_mean','delta_macro_f1','delta_macro_f1_pct',
                   'balanced_accuracy_mean','delta_balanced_accuracy','ganho']].sort_values('delta_macro_f1', ascending=False))

openSMILE isolado — Macro F1: 0.4680 | Balanced Acc: 0.4843 | Accuracy: 0.5203


### Delta vs openSMILE isolado

,scenario,best_model,macro_f1_mean,delta_macro_f1,delta_macro_f1_pct,balanced_accuracy_mean,delta_balanced_accuracy,ganho
0,10_os_allfp_rawpeaks,HistGradientBoosting,0.482592,0.014576,3.114322,0.481548,-0.002758,melhora em Macro F1
7,7_os_allfp,HistGradientBoosting,0.481370,0.013354,2.853233,0.478903,-0.005403,melhora em Macro F1
6,6_os_fpband,HistGradientBoosting,0.478336,0.010320,2.204952,0.475535,-0.008771,melhora em Macro F1
5,5_os_fprank,HistGradientBoosting,0.475220,0.007204,1.539168,0.474997,-0.009309,melhora pequena
9,9_os_rawpeaks,HistGradientBoosting,0.474485,0.006469,1.382110,0.473122,-0.011184,melhora pequena
4,4_allfp_only,LogisticRegression,0.441557,-0.026460,-5.653603,0.458988,-0.025318,sem melhora relevante
2,2_fprank_only,LogisticRegression,0.431237,-0.036780,-7.858616,0.445173,-0.039132,sem melhora relevante
8,8_rawpeaks_only,LogisticRegression,0.407382,-0.060635,-12.955636,0.427278,-0.057028,sem melhora relevante
3,3_fpband_only,LogisticRegression,0.404186,-0.063831,-13.638534,0.424547,-0.059759,sem melhora relevante


## 9. Métricas por quadrante

In [16]:
# comparação de F1 por quadrante: openSMILE | melhor FP isolado | melhor fusão
per_class_agg = (
    ok_df.groupby(['scenario', 'model'])
    [per_class_f1_cols].mean()
    .reset_index()
) if per_class_f1_cols else pd.DataFrame()

if not per_class_agg.empty:
    def get_best_per_class(scenario_name):
        sub = agg_df[agg_df['scenario'] == scenario_name]
        if sub.empty: return None
        best_m = sub.sort_values('macro_f1_mean', ascending=False).iloc[0]['model']
        row = per_class_agg[(per_class_agg['scenario']==scenario_name) & (per_class_agg['model']==best_m)]
        if row.empty: return None
        return row.iloc[0]

    best_fusion_scenario = delta_df.sort_values('delta_macro_f1', ascending=False).iloc[0]['scenario'] if not delta_df.empty else None
    best_fp_only_scenario = (
        best_by_scenario[best_by_scenario['scenario'].isin(['2_fprank_only','3_fpband_only','4_allfp_only'])]
        .sort_values('macro_f1_mean', ascending=False)
    )
    best_fp_scenario = best_fp_only_scenario.iloc[0]['scenario'] if not best_fp_only_scenario.empty else None

    compare_rows = []
    for scen_name, label in [
        ('1_opensmile_only', 'openSMILE'),
        (best_fp_scenario,   'Melhor FP isolado'),
        (best_fusion_scenario, 'Melhor Fusão'),
    ]:
        if scen_name is None: continue
        row = get_best_per_class(scen_name)
        if row is not None:
            compare_rows.append({'fonte': label, 'cenario': scen_name, **{c: row[c] for c in per_class_f1_cols if c in row.index}})

    per_class_compare = pd.DataFrame(compare_rows)
    display(Markdown('### F1 por Quadrante: openSMILE vs FP isolado vs Melhor Fusão'))
    display(per_class_compare)
else:
    per_class_compare = pd.DataFrame()
    print('[INFO] Métricas por classe não disponíveis')

### F1 por Quadrante: openSMILE vs FP isolado vs Melhor Fusão

,fonte,cenario,Q1_f1,Q2_f1,Q3_f1,Q4_f1
0,openSMILE,1_opensmile_only,0.648890,0.405093,0.506419,0.311665
1,Melhor FP isolado,4_allfp_only,0.624129,0.374020,0.483990,0.284088
2,Melhor Fusão,10_os_allfp_rawpeaks,0.760549,0.360441,0.599721,0.209657


## 10. Matrizes de confusão (OOF)

In [17]:
def build_oof_cm(scenario_name: str, model_name: str, y_true: np.ndarray) -> Optional[pd.DataFrame]:
    oof = oof_store.get(scenario_name, {}).get(model_name)
    if oof is None or (oof == '').all():
        return None
    mask = (oof != '') & (y_true != '')
    cm   = confusion_matrix(y_true[mask], oof[mask], labels=labels)
    short_labels = [cfg.QUADRANT_SHORT.get(l, l) for l in labels]
    return pd.DataFrame(cm, index=short_labels, columns=short_labels)


def get_best_model_for(scenario_name: str) -> str:
    sub = agg_df[agg_df['scenario'] == scenario_name]
    if sub.empty: return list(models.keys())[0]
    return sub.sort_values('macro_f1_mean', ascending=False).iloc[0]['model']


y_true_all = y_all.values

cm_os_best_model   = get_best_model_for('1_opensmile_only')
cm_fp_best_scenario = best_fp_scenario or '2_fprank_only'
cm_fp_best_model    = get_best_model_for(cm_fp_best_scenario)
cm_fus_best_scenario = best_fusion_scenario or '7_os_allfp'
cm_fus_best_model    = get_best_model_for(cm_fus_best_scenario)

cm_os  = build_oof_cm('1_opensmile_only',   cm_os_best_model,  y_true_all)
cm_fp  = build_oof_cm(cm_fp_best_scenario,  cm_fp_best_model,  y_true_all)
cm_fus = build_oof_cm(cm_fus_best_scenario, cm_fus_best_model, y_true_all)

cm_records = []
for cm_df, name, model in [
    (cm_os,  '1_opensmile_only',    cm_os_best_model),
    (cm_fp,  cm_fp_best_scenario,   cm_fp_best_model),
    (cm_fus, cm_fus_best_scenario,  cm_fus_best_model),
]:
    if cm_df is not None:
        tmp = cm_df.reset_index().melt(id_vars='index', var_name='predicted', value_name='count')
        tmp.columns = ['true','predicted','count']
        tmp.insert(0, 'scenario', name)
        tmp.insert(1, 'model', model)
        cm_records.append(tmp)

for cm_df, title in [
    (cm_os,  f'openSMILE ({cm_os_best_model})'),
    (cm_fp,  f'FP isolado — {slabel(cm_fp_best_scenario)} ({cm_fp_best_model})'),
    (cm_fus, f'Melhor Fusão — {slabel(cm_fus_best_scenario)} ({cm_fus_best_model})'),
]:
    if cm_df is not None:
        display(Markdown(f'### Matriz de Confusão: {title}'))
        display(cm_df)

### Matriz de Confusão: openSMILE (LogisticRegression)

,Q1,Q2,Q3,Q4
Q1,1850,618,302,447
Q2,281,481,147,110
Q3,145,165,715,357
Q4,202,81,266,339


### Matriz de Confusão: FP isolado — 4. All FP (LogisticRegression)

,Q1,Q2,Q3,Q4
Q1,1727,687,334,469
Q2,296,450,121,152
Q3,160,126,692,404
Q4,135,118,311,324


### Matriz de Confusão: Melhor Fusão — 10. OS+AllFP+Raw (HistGradientBoosting)

,Q1,Q2,Q3,Q4
Q1,2810,121,216,70
Q2,586,266,136,31
Q3,354,48,885,95
Q4,421,19,318,130


## 11. Importância das features por origem

In [18]:
importance_df = pd.DataFrame(importance_records)
imp_by_origin = pd.DataFrame()

if not importance_df.empty:
    importance_df['origin'] = importance_df['feature'].map(feature_origin)

    imp_by_origin = (
        importance_df.groupby(['scenario','model','origin'])['importance']
        .sum()
        .reset_index()
    )
    total_imp = imp_by_origin.groupby(['scenario','model'])['importance'].transform('sum')
    imp_by_origin['importance_pct'] = 100 * imp_by_origin['importance'] / total_imp.replace(0, np.nan)

    display(Markdown('### Importância por Origem — cenário 7 (OS + All FP)'))
    sub = imp_by_origin[imp_by_origin['scenario'] == '7_os_allfp'].sort_values('importance_pct', ascending=False)
    display(sub)

    # Top 20 features — cenário 7
    display(Markdown('### Top 20 Features — Melhor Cenário de Fusão'))
    tree_models = ['ExtraTrees','RandomForest','GradientBoosting','HistGradientBoosting']
    top20_src = importance_df[
        (importance_df['scenario'] == cm_fus_best_scenario) &
        (importance_df['model'].isin(tree_models))
    ]
    if top20_src.empty:
        top20_src = importance_df[importance_df['scenario'] == cm_fus_best_scenario]
    top20 = (
        top20_src.groupby('feature')['importance'].mean()
        .sort_values(ascending=False)
        .head(20)
        .reset_index()
    )
    top20['origin'] = top20['feature'].map(feature_origin)
    display(top20)
else:
    print('[INFO] Nenhuma importância de feature registrada')

### Importância por Origem — cenário 7 (OS + All FP)

,scenario,model,origin,importance,importance_pct
64,7_os_allfp,RandomForest,openSMILE,0.608819,60.881869
61,7_os_allfp,LogisticRegression,openSMILE,147.424573,59.822354
67,7_os_allfp,RidgeClassifier,openSMILE,64.772938,55.457477
58,7_os_allfp,ExtraTrees,openSMILE,0.536931,53.693136
56,7_os_allfp,ExtraTrees,Fingerprint Band,0.289724,28.972438
65,7_os_allfp,RidgeClassifier,Fingerprint Band,29.185834,24.988410
62,7_os_allfp,RandomForest,Fingerprint Band,0.247853,24.785344
59,7_os_allfp,LogisticRegression,Fingerprint Band,59.996274,24.345455
66,7_os_allfp,RidgeClassifier,Fingerprint Rank,22.838711,19.554112
57,7_os_allfp,ExtraTrees,Fingerprint Rank,0.173344,17.334426


### Top 20 Features — Melhor Cenário de Fusão

,feature,importance,origin
0,fpband__energy_norm_high,0.006536,Fingerprint Band
1,opensmile__audspec_lengthL1norm_sma_amean,0.004277,openSMILE
2,opensmile__pcm_fftMag_mfcc_sma[1]_amean,0.004138,openSMILE
3,fprank__fp_magnitude_mean_norm_block,0.004113,Fingerprint Rank
4,fpband__fp_high_top_5_magnitude_norm_block,0.003729,Fingerprint Band
5,fpband__fp_high_top_8_magnitude_norm_block,0.003716,Fingerprint Band
6,opensmile__logHNR_sma_stddev,0.003713,openSMILE
7,opensmile__pcm_fftMag_spectralFlux_sma_amean,0.003672,openSMILE
8,fpband__fp_high_top_7_magnitude_norm_block,0.003456,Fingerprint Band
9,fpband__fp_high_top_9_magnitude_norm_block,0.003400,Fingerprint Band


## 12. Gráficos

In [19]:
PLOTLY_TEMPLATE = 'plotly_white'

SCENARIO_TYPE = {
    '1_opensmile_only':     'openSMILE',
    '2_fprank_only':        'Apenas FP',
    '3_fpband_only':        'Apenas FP',
    '4_allfp_only':         'Apenas FP',
    '5_os_fprank':          'Fusao OS+FP',
    '6_os_fpband':          'Fusao OS+FP',
    '7_os_allfp':           'Fusao OS+FP',
    '8_rawpeaks_only':      'Apenas FP',
    '9_os_rawpeaks':        'Fusao OS+FP',
    '10_os_allfp_rawpeaks': 'Fusao OS+FP',
}

SCENARIO_TYPE_COLORS = {
    'openSMILE':    '#2196F3',
    'Apenas FP':    '#FF9800',
    'Fusao OS+FP':  '#4CAF50',
}


def stype(s: str) -> str:
    return SCENARIO_TYPE.get(s, 'outros')


def save_plot(fig, name: str, cfg) -> None:
    fig.update_layout(template=PLOTLY_TEMPLATE)
    if cfg.export_html:
        path = cfg.figures_dir() / f'{name}.html'
        fig.write_html(str(path), include_plotlyjs='cdn')
        print(f'Salvo: {path.name}')
    if cfg.show_figures:
        fig.show()

In [20]:
# Grafico 1: Macro F1 por cenario
plot_f1 = best_by_scenario.copy().sort_values('scenario')
plot_f1['label'] = plot_f1['scenario'].map(slabel)
plot_f1['tipo'] = plot_f1['scenario'].map(stype)

label_order_f1 = (
    plot_f1.groupby('label')['macro_f1_mean'].mean()
    .sort_values(ascending=True)
    .index.tolist()
)

fig_f1 = px.bar(
    plot_f1,
    x='macro_f1_mean',
    y='label',
    color='tipo',
    orientation='h',
    error_x='macro_f1_std',
    text=plot_f1['macro_f1_mean'].map('{:.3f}'.format),
    hover_data={
        'best_model': True,
        'n_features': True,
        'balanced_accuracy_mean': ':.3f',
        'accuracy_mean': ':.3f',
        'macro_f1_std': ':.4f',
        'tipo': False,
    },
    color_discrete_map=SCENARIO_TYPE_COLORS,
    category_orders={'label': label_order_f1},
    title='Macro F1 por Cenario — Classificacao de Quadrantes (GroupKFold n=5)',
    labels={'macro_f1_mean': 'Macro F1 medio', 'label': '', 'tipo': 'Tipo'},
)
fig_f1.update_traces(textposition='outside')
fig_f1.add_vline(
    x=os_mf1, line_dash='dash', line_color='navy', line_width=1.5,
    annotation_text=f'openSMILE={os_mf1:.3f}', annotation_position='top right',
)
fig_f1.update_layout(
    template=PLOTLY_TEMPLATE, height=460, legend_title='Tipo de Cenario',
    margin=dict(l=10, r=80, t=60, b=10),
)

save_plot(fig_f1, 'fig_macro_f1_por_cenario', cfg)

Salvo: fig_macro_f1_por_cenario.html


In [21]:
# Grafico 2: Balanced Accuracy por cenario
plot_ba = best_by_scenario.copy().sort_values('scenario')
plot_ba['label'] = plot_ba['scenario'].map(slabel)
plot_ba['tipo'] = plot_ba['scenario'].map(stype)

label_order_ba = (
    plot_ba.groupby('label')['balanced_accuracy_mean'].mean()
    .sort_values(ascending=True)
    .index.tolist()
)

fig_ba = px.bar(
    plot_ba,
    x='balanced_accuracy_mean',
    y='label',
    color='tipo',
    orientation='h',
    error_x='balanced_accuracy_std' if 'balanced_accuracy_std' in plot_ba.columns else None,
    text=plot_ba['balanced_accuracy_mean'].map('{:.3f}'.format),
    hover_data={
        'best_model': True,
        'n_features': True,
        'macro_f1_mean': ':.3f',
        'tipo': False,
    },
    color_discrete_map=SCENARIO_TYPE_COLORS,
    category_orders={'label': label_order_ba},
    title='Balanced Accuracy por Cenario (GroupKFold n=5)',
    labels={'balanced_accuracy_mean': 'Balanced Accuracy', 'label': '', 'tipo': 'Tipo'},
)
fig_ba.update_traces(textposition='outside')
fig_ba.add_vline(
    x=os_ba, line_dash='dash', line_color='navy', line_width=1.5,
    annotation_text=f'openSMILE={os_ba:.3f}', annotation_position='top right',
)
fig_ba.update_layout(
    template=PLOTLY_TEMPLATE, height=460, legend_title='Tipo de Cenario',
    margin=dict(l=10, r=80, t=60, b=10),
)

save_plot(fig_ba, 'fig_balanced_accuracy_por_cenario', cfg)

Salvo: fig_balanced_accuracy_por_cenario.html


In [22]:
# Grafico 3: Delta Macro F1 vs openSMILE
if not delta_df.empty:
    plot_delta = delta_df.copy().sort_values('scenario')
    plot_delta['label'] = plot_delta['scenario'].map(slabel)
    plot_delta['cor'] = plot_delta['delta_macro_f1'].apply(
        lambda d: 'Melhorou' if d > 0 else 'Piorou'
    )

    label_order_delta = (
        plot_delta.groupby('label')['delta_macro_f1'].mean()
        .sort_values(ascending=False)
        .index.tolist()
    )

    fig_delta = px.bar(
        plot_delta,
        x='delta_macro_f1',
        y='label',
        color='cor',
        orientation='h',
        text=plot_delta['delta_macro_f1'].map('{:+.4f}'.format),
        hover_data={
            'best_model': True,
            'macro_f1_mean': ':.4f',
            'delta_macro_f1_pct': ':.2f',
            'delta_balanced_accuracy': ':.4f',
            'ganho': True,
            'cor': False,
        },
        color_discrete_map={'Melhorou': '#4CAF50', 'Piorou': '#F44336'},
        category_orders={'label': label_order_delta},
        title='Delta Macro F1 — Cenarios de Fusao vs openSMILE Isolado (positivo = melhor)',
        labels={
            'delta_macro_f1': 'delta Macro F1 (fusao - openSMILE)',
            'label': '',
            'cor': 'Impacto',
        },
    )
    fig_delta.update_traces(textposition='outside')
    fig_delta.add_vline(x=0, line_dash='dash', line_color='black', line_width=1.5)
    fig_delta.update_layout(
        template=PLOTLY_TEMPLATE, height=460, legend_title='Impacto',
        margin=dict(l=10, r=80, t=60, b=10),
    )

    save_plot(fig_delta, 'fig_delta_macro_f1', cfg)

Salvo: fig_delta_macro_f1.html


In [23]:
# Grafico 4: F1 por quadrante (openSMILE vs FP isolado vs Melhor Fusao)
if not per_class_compare.empty:
    short_cols = [c for c in per_class_f1_cols if 'f1' in c]
    plot_per_class = per_class_compare.melt(
        id_vars='fonte', value_vars=short_cols,
        var_name='quadrante', value_name='f1_score'
    )
    plot_per_class['quadrante'] = plot_per_class['quadrante'].str.replace('_f1', '', regex=False)

    fig_pc = px.bar(
        plot_per_class,
        x='quadrante',
        y='f1_score',
        color='fonte',
        barmode='group',
        text=plot_per_class['f1_score'].map(lambda v: f'{v:.3f}' if pd.notna(v) else ''),
        hover_data={'f1_score': ':.4f'},
        color_discrete_sequence=['#2196F3', '#FF9800', '#4CAF50'],
        title='F1-score por Quadrante: openSMILE vs FP isolado vs Melhor Fusao',
        labels={'f1_score': 'F1-score', 'quadrante': 'Quadrante', 'fonte': 'Representacao'},
    )
    fig_pc.update_traces(textposition='outside')
    fig_pc.update_layout(
        template=PLOTLY_TEMPLATE, height=450, legend_title='Representacao',
        yaxis=dict(range=[0, 1.1]),
        margin=dict(l=10, r=10, t=60, b=10),
    )

    save_plot(fig_pc, 'fig_f1_por_quadrante', cfg)

Salvo: fig_f1_por_quadrante.html


In [24]:
# Graficos 5-7: matrizes de confusao OOF (uma por cenario)
for cm_df, title, name in [
    (cm_os,  f'openSMILE — {cm_os_best_model}',
             'fig_cm_opensmile'),
    (cm_fp,  f'{slabel(cm_fp_best_scenario)} — {cm_fp_best_model}',
             'fig_cm_fp_isolado'),
    (cm_fus, f'Fusao — {slabel(cm_fus_best_scenario)} — {cm_fus_best_model}',
             'fig_cm_fusao'),
]:
    if cm_df is None or cm_df.empty:
        print(f'[INFO] Sem dados de CM para: {title}')
        continue
    cm_arr = cm_df.values.astype(float)
    row_sum = cm_arr.sum(axis=1, keepdims=True)
    cm_norm = cm_arr / np.where(row_sum == 0, 1, row_sum)
    cm_norm_df = pd.DataFrame(cm_norm, index=cm_df.index, columns=cm_df.columns)

    fig_cm = px.imshow(
        cm_norm_df,
        text_auto='.2f',
        aspect='auto',
        color_continuous_scale='Blues',
        title=f'Matriz de Confusao (OOF, normalizada) — {title}',
    )
    fig_cm.update_layout(
        xaxis_title='Predito',
        yaxis_title='Real',
        height=420,
        margin=dict(l=10, r=10, t=60, b=10),
    )

    save_plot(fig_cm, name, cfg)

Salvo: fig_cm_opensmile.html


Salvo: fig_cm_fp_isolado.html


Salvo: fig_cm_fusao.html


In [25]:
# Grafico 8: importancia por origem (cenario 7 — OS+AllFP)
if not imp_by_origin.empty:
    sub7 = imp_by_origin[
        (imp_by_origin['scenario'] == '7_os_allfp') &
        (imp_by_origin['model'].isin(['ExtraTrees','RandomForest','GradientBoosting','HistGradientBoosting']))
    ]
    if sub7.empty:
        sub7 = imp_by_origin[imp_by_origin['scenario'] == '7_os_allfp']
    if not sub7.empty:
        grp_imp = (
            sub7.groupby('origin')['importance_pct'].mean()
            .reset_index()
            .sort_values('importance_pct', ascending=False)
        )

        fig_imp = px.bar(
            grp_imp,
            x='origin',
            y='importance_pct',
            color='origin',
            text=grp_imp['importance_pct'].map('{:.1f}%'.format),
            hover_data={'importance_pct': ':.2f'},
            color_discrete_map=ORIGIN_COLORS,
            title='Importancia por Origem — OS + All Fingerprints (ExtraTrees + RF + GB)',
            labels={'importance_pct': 'Importancia relativa (%)', 'origin': 'Origem'},
        )
        fig_imp.update_traces(textposition='outside', showlegend=False)
        fig_imp.update_layout(
            template=PLOTLY_TEMPLATE, height=420,
            margin=dict(l=10, r=10, t=60, b=10),
        )

        save_plot(fig_imp, 'fig_importancia_por_origem', cfg)

Salvo: fig_importancia_por_origem.html


In [26]:
# Grafico 9: top 20 features — melhor cenario de fusao
if not importance_df.empty and not top20.empty:
    top20_plot = top20.copy()
    top20_plot['short'] = (
        top20_plot['feature']
        .str.replace('opensmile__', 'os:', regex=False)
        .str.replace('fprank__', 'fpr:', regex=False)
        .str.replace('fpband__', 'fpb:', regex=False)
        .str.replace('rawpeaks_rank__', 'rpr:', regex=False)
        .str.replace('rawpeaks_band__', 'rpb:', regex=False)
        .str[:60]
    )

    fig_top = px.bar(
        top20_plot.sort_values('importance', ascending=True),
        x='importance',
        y='short',
        color='origin',
        orientation='h',
        text=top20_plot.sort_values('importance', ascending=True)['importance'].map('{:.4f}'.format),
        hover_data={
            'feature': True,
            'importance': ':.5f',
            'origin': True,
            'short': False,
        },
        color_discrete_map=ORIGIN_COLORS,
        title=f'Top 20 Features — {slabel(cm_fus_best_scenario)} ({cm_fus_best_model})',
        labels={
            'importance': 'Importancia media',
            'short': '',
            'origin': 'Origem',
        },
    )
    fig_top.update_traces(textposition='outside')
    fig_top.update_layout(
        template=PLOTLY_TEMPLATE, height=620, legend_title='Origem',
        margin=dict(l=10, r=80, t=60, b=10),
    )

    save_plot(fig_top, 'fig_top20_features_fusao', cfg)

Salvo: fig_top20_features_fusao.html


In [27]:
# Grafico 10: visao geral multi-metrica por cenario
metrics_cfg = [
    ('macro_f1_mean',           'Macro F1',           'macro_f1_std'),
    ('balanced_accuracy_mean',  'Balanced Accuracy',  'balanced_accuracy_std'),
    ('accuracy_mean',           'Accuracy',           'accuracy_std'),
    ('weighted_f1_mean',        'Weighted F1',        'weighted_f1_std'),
]

multi_frames = []
for metric_col, metric_label, std_col in metrics_cfg:
    if metric_col not in best_by_scenario.columns:
        continue
    sub = best_by_scenario[['scenario', 'best_model', 'n_features', metric_col]].copy()
    std_vals = best_by_scenario[std_col] if std_col in best_by_scenario.columns else None
    sub['valor_std'] = std_vals if std_vals is not None else 0.0
    sub = sub.rename(columns={metric_col: 'valor'})
    sub['metrica'] = metric_label
    sub['label'] = sub['scenario'].map(slabel)
    sub['tipo'] = sub['scenario'].map(stype)
    multi_frames.append(sub)

if multi_frames:
    multi_df = pd.concat(multi_frames, ignore_index=True)

    fig_multi = px.bar(
        multi_df.sort_values('scenario'),
        x='label',
        y='valor',
        error_y='valor_std',
        color='tipo',
        facet_col='metrica',
        barmode='group',
        text=multi_df.sort_values('scenario')['valor'].map('{:.3f}'.format),
        hover_data={'best_model': True, 'n_features': True, 'tipo': False, 'valor_std': ':.4f'},
        color_discrete_map=SCENARIO_TYPE_COLORS,
        title='Visao Geral Multi-Metrica por Cenario (melhor modelo de cada cenario)',
        labels={'valor': 'Valor', 'label': 'Cenario', 'tipo': 'Tipo'},
    )
    fig_multi.update_traces(
        texttemplate='%{text}',
        textposition='outside',
        textfont_size=8,
    )
    fig_multi.update_layout(
        template=PLOTLY_TEMPLATE,
        height=500,
        legend_title='Tipo de Cenario',
        xaxis_tickangle=-40,
        margin=dict(l=10, r=10, t=80, b=60),
    )
    fig_multi.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1]))
    fig_multi.update_yaxes(matches=None)

    save_plot(fig_multi, 'fig_multimetrica_cenarios', cfg)

Salvo: fig_multimetrica_cenarios.html


## 13. Exportacao dos resultados

In [28]:
out = cfg.output_dir()

fold_df.to_csv(out / 'fusion_quadrants_fold_results.csv', index=False)
agg_df.to_csv(out / 'fusion_quadrants_all_results.csv', index=False)
best_by_scenario.to_csv(out / 'fusion_quadrants_best_by_scenario.csv', index=False)

if not delta_df.empty:
    delta_df.to_csv(out / 'fusion_quadrants_delta_vs_opensmile.csv', index=False)

if not per_class_compare.empty:
    per_class_compare.to_csv(out / 'fusion_quadrants_per_class_f1.csv', index=False)

# relatório por classe (sklearn classification_report)
cr_rows = []
for scenario_name in ['1_opensmile_only', cm_fp_best_scenario, cm_fus_best_scenario]:
    best_m = get_best_model_for(scenario_name)
    oof = oof_store.get(scenario_name, {}).get(best_m)
    if oof is not None:
        mask = oof != ''
        cr = classification_report(y_true_all[mask], oof[mask], labels=labels, output_dict=True, zero_division=0)
        for class_name, vals in cr.items():
            if isinstance(vals, dict):
                cr_rows.append({'scenario': scenario_name, 'model': best_m, 'class': class_name, **vals})

if cr_rows:
    pd.DataFrame(cr_rows).to_csv(out / 'fusion_quadrants_class_report.csv', index=False)

if cm_records:
    pd.concat(cm_records, ignore_index=True).to_csv(out / 'fusion_quadrants_confusion_matrices.csv', index=False)

if not importance_df.empty:
    importance_df.to_csv(out / 'fusion_quadrants_feature_importance.csv', index=False)
if not imp_by_origin.empty:
    imp_by_origin.to_csv(out / 'fusion_quadrants_importance_by_origin.csv', index=False)

print('Exportado para:', out)
for f in sorted(out.glob('*.csv')):
    print(f'  {f.name}')

print('Figuras HTML:', cfg.figures_dir())
for f in sorted(cfg.figures_dir().glob('*.html')):
    print(f'  {f.name}')

Exportado para: C:\dev\python\TCC Ajustado\Dados\pycaret_fusion_quadrants_validation
  fusion_quadrants_all_results.csv
  fusion_quadrants_best_by_scenario.csv
  fusion_quadrants_class_report.csv
  fusion_quadrants_confusion_matrices.csv
  fusion_quadrants_delta_vs_opensmile.csv
  fusion_quadrants_feature_importance.csv
  fusion_quadrants_fold_results.csv
  fusion_quadrants_importance_by_origin.csv
  fusion_quadrants_per_class_f1.csv
Figuras HTML: C:\dev\python\TCC Ajustado\Dados\pycaret_fusion_quadrants_validation\figures
  fig_balanced_accuracy_por_cenario.html
  fig_cm_fp_isolado.html
  fig_cm_fusao.html
  fig_cm_opensmile.html
  fig_delta_macro_f1.html
  fig_f1_por_quadrante.html
  fig_importancia_por_origem.html
  fig_macro_f1_por_cenario.html
  fig_multimetrica_cenarios.html
  fig_top20_features_fusao.html


## 14. Síntese final

In [29]:
def sintetizar_quadrantes(
    best_by_scenario, delta_df, per_class_compare, imp_by_origin,
    os_mf1, os_ba, per_class_f1_cols, cfg
) -> str:
    lines = [
        '# Síntese: Fusão openSMILE + Fingerprints — Classificação de Quadrantes',
        '',
        '> Esta síntese foi gerada automaticamente com base nos resultados da validação GroupKFold(n=5) por song_id.',
        '> O objetivo foi avaliar empiricamente se os fingerprints acústicos complementam o openSMILE',
        '> na classificação dos quadrantes emocionais Q1–Q4 (espaço Valence × Arousal).',
        '> A conclusão é neutra e baseada nos resultados observados, sem hipótese prévia de melhora.',
        '',
        f'**Baseline openSMILE isolado:** Macro F1 = {os_mf1:.4f} | Balanced Accuracy = {os_ba:.4f}',
        '',
    ]

    # 1. O fingerprint melhora o openSMILE?
    lines.append('## 1. O fingerprint melhora a classificação de quadrantes em relação ao openSMILE?')
    if delta_df.empty:
        lines.append('_Sem dados de comparação disponíveis._')
    else:
        ganhos = delta_df['ganho'].value_counts()
        n_con  = ganhos.get('melhora consistente', 0)
        n_f1   = ganhos.get('melhora em Macro F1', 0)
        n_ba   = ganhos.get('melhora em Balanced Accuracy', 0)
        n_peq  = ganhos.get('melhora pequena', 0)
        n_sem  = ganhos.get('sem melhora relevante', 0)
        total  = len(delta_df)

        best_fusion = delta_df.sort_values('delta_macro_f1', ascending=False).iloc[0]
        lines.append(f'- {total} cenários de fusão avaliados.')
        if n_con > 0:
            lines.append(f'- **Melhora consistente** (Macro F1 + Balanced Acc) em {n_con}/{total} cenários.')
        if n_f1 > 0:
            lines.append(f'- Melhora principalmente em Macro F1 em {n_f1} cenários.')
        if n_ba > 0:
            lines.append(f'- Melhora principalmente em Balanced Accuracy em {n_ba} cenários.')
        if n_peq > 0:
            lines.append(f'- Melhora pequena (< 1pp) em {n_peq} cenários.')
        if n_sem > 0:
            lines.append(f'- Sem melhora relevante em {n_sem} cenários.')
        lines.append(f'- **Melhor cenário:** {slabel(best_fusion["scenario"])} '
                     f'| Δ Macro F1 = {best_fusion["delta_macro_f1"]:+.4f} '
                     f'| Δ Balanced Acc = {best_fusion["delta_balanced_accuracy"]:+.4f}')
    lines.append('')

    # 2. Por quadrante
    lines.append('## 2. A melhora acontece em todos os quadrantes ou apenas em alguns?')
    if not per_class_compare.empty:
        f1_cols = [c for c in per_class_f1_cols if c.endswith('_f1')]
        os_row  = per_class_compare[per_class_compare['fonte'] == 'openSMILE']
        fus_row = per_class_compare[per_class_compare['fonte'] == 'Melhor Fusão']
        if not os_row.empty and not fus_row.empty:
            for fc in f1_cols:
                q   = fc.replace('_f1', '')
                val_os  = float(os_row[fc].values[0]) if fc in os_row.columns else np.nan
                val_fus = float(fus_row[fc].values[0]) if fc in fus_row.columns else np.nan
                if not np.isnan(val_os) and not np.isnan(val_fus):
                    delta = val_fus - val_os
                    sinal = '+' if delta > 0 else ''
                    lines.append(f'  - **{q}**: openSMILE {val_os:.3f} → Fusão {val_fus:.3f} (Δ {sinal}{delta:.3f})')
        else:
            lines.append('  _Dados por quadrante incompletos._')
    else:
        lines.append('  _Métricas por quadrante não disponíveis._')
    lines.append('')

    # 3. Qual tipo de fingerprint ajuda mais?
    lines.append('## 3. Qual tipo de fingerprint ajuda mais: ranking geral ou por bandas?')
    fp_types = best_by_scenario[best_by_scenario['scenario'].isin(['2_fprank_only','3_fpband_only'])]
    if len(fp_types) >= 2:
        rank_f1 = fp_types[fp_types['scenario']=='2_fprank_only']['macro_f1_mean'].values
        band_f1 = fp_types[fp_types['scenario']=='3_fpband_only']['macro_f1_mean'].values
        if len(rank_f1) and len(band_f1):
            if rank_f1[0] > band_f1[0]:
                lines.append(f'  - **Fingerprint Rank** supera Band isolado: F1 {rank_f1[0]:.4f} vs {band_f1[0]:.4f}.')
            elif band_f1[0] > rank_f1[0]:
                lines.append(f'  - **Fingerprint Band** supera Rank isolado: F1 {band_f1[0]:.4f} vs {rank_f1[0]:.4f}.')
            else:
                lines.append('  - Fingerprint Rank e Band apresentam F1 equivalente isolados.')
    else:
        lines.append('  _Dados insuficientes para comparar tipos._')
    lines.append('')

    # 4. Macro F1 e Balanced Accuracy
    lines.append('## 4. A fusão melhora Macro F1 e Balanced Accuracy?')
    if not delta_df.empty:
        best_7 = delta_df[delta_df['scenario'] == '7_os_allfp']
        if not best_7.empty:
            r = best_7.iloc[0]
            lines.append(f'  - Cenário OS+AllFP: Δ Macro F1 = {r["delta_macro_f1"]:+.4f}, '
                         f'Δ Balanced Acc = {r["delta_balanced_accuracy"]:+.4f}')
            lines.append(f'  - Classificação: **{r["ganho"]}**')
    lines.append('')

    # 5. Importância das features
    lines.append('## 5. As features de fingerprint aparecem entre as mais importantes?')
    if not imp_by_origin.empty:
        sub7 = imp_by_origin[imp_by_origin['scenario'] == '7_os_allfp']
        if not sub7.empty:
            grp = sub7.groupby('origin')['importance_pct'].mean().sort_values(ascending=False)
            for origin, pct in grp.items():
                lines.append(f'  - {origin}: {pct:.1f}% da importância total')
            fp_total = grp.drop('openSMILE', errors='ignore').sum()
            if fp_total > 30:
                lines.append(f'  - **Sim**: fingerprints somam {fp_total:.1f}% da importância no cenário OS+AllFP.')
            else:
                lines.append(f'  - Os fingerprints somam {fp_total:.1f}%: openSMILE ainda domina a importância.')
    else:
        lines.append('  _Importância de features não disponível._')
    lines.append('')

    # 6. Complementaridade ou redundância?
    lines.append('## 6. A fusão indica complementaridade ou redundância?')
    if not delta_df.empty:
        best_fus = delta_df.sort_values('delta_macro_f1', ascending=False).iloc[0]
        d = best_fus['delta_macro_f1']
        if d > 0.015:
            lines.append('  - **Complementaridade**: o melhor cenário de fusão melhora o Macro F1 '
                         f'em {d*100:+.2f}pp, sugerindo que os fingerprints trazem informação adicional.')
        elif d > 0.003:
            lines.append('  - **Complementaridade parcial**: ganho pequeno mas positivo '
                         f'(Δ Macro F1 = {d*100:+.2f}pp), sugerindo sobreposição parcial com openSMILE.')
        elif d > -0.005:
            lines.append(f'  - **Resultado neutro**: Δ Macro F1 = {d*100:+.2f}pp. '
                         'Os fingerprints parecem capturar informação parcialmente redundante com o openSMILE.')
        else:
            lines.append(f'  - **Redundância ou ruído**: Δ Macro F1 = {d*100:+.2f}pp. '
                         'A fusão piora o desempenho, indicando que os fingerprints introduzem ruído.')
    lines.append('')

    # 7. Contribuição suficiente para defender?
    lines.append('## 7. O resultado é suficiente para defender que o fingerprint contribui para a classificação emocional?')
    if not delta_df.empty:
        n_positivo = (delta_df['delta_macro_f1'] > 0.003).sum()
        total_cen  = len(delta_df)
        if n_positivo >= total_cen * 0.5:
            lines.append(f'  - **Sim, com ressalvas**: {n_positivo}/{total_cen} cenários de fusão apresentam ganho positivo.')
            lines.append('  - A evidência sugere que o fingerprint agrega informação, mas o ganho não é sempre expressivo.')
            lines.append('  - Recomenda-se reportar tanto os casos de melhora quanto os neutros para objetividade acadêmica.')
        else:
            lines.append(f'  - **Evidência limitada**: apenas {n_positivo}/{total_cen} cenários apresentam ganho > 0.3pp.')
            lines.append('  - Não há base empírica sólida para afirmar que o fingerprint melhora sistematicamente a classificação.')
            lines.append('  - Os fingerprints podem ser utilizados como análise exploratória ou secundária ao openSMILE.')
    lines.append('')
    lines += [
        '---',
        f'_Gerado em: {pd.Timestamp.now().strftime("%Y-%m-%d %H:%M")} | Validação: GroupKFold(n=5) por song_id_',
    ]
    return '\n'.join(lines)


synthesis = sintetizar_quadrantes(
    best_by_scenario, delta_df, per_class_compare, imp_by_origin,
    os_mf1, os_ba, per_class_f1_cols, cfg
)
display(Markdown(synthesis))

with open(cfg.output_dir() / 'fusion_quadrants_interpretation_summary.md', 'w', encoding='utf-8') as f:
    f.write(synthesis)
print('Síntese exportada:', cfg.output_dir() / 'fusion_quadrants_interpretation_summary.md')

# Síntese: Fusão openSMILE + Fingerprints — Classificação de Quadrantes

> Esta síntese foi gerada automaticamente com base nos resultados da validação GroupKFold(n=5) por song_id.
> O objetivo foi avaliar empiricamente se os fingerprints acústicos complementam o openSMILE
> na classificação dos quadrantes emocionais Q1–Q4 (espaço Valence × Arousal).
> A conclusão é neutra e baseada nos resultados observados, sem hipótese prévia de melhora.

**Baseline openSMILE isolado:** Macro F1 = 0.4680 | Balanced Accuracy = 0.4843

## 1. O fingerprint melhora a classificação de quadrantes em relação ao openSMILE?
- 9 cenários de fusão avaliados.
- Melhora principalmente em Macro F1 em 3 cenários.
- Melhora pequena (< 1pp) em 2 cenários.
- Sem melhora relevante em 4 cenários.
- **Melhor cenário:** 10. OS+AllFP+Raw | Δ Macro F1 = +0.0146 | Δ Balanced Acc = -0.0028

## 2. A melhora acontece em todos os quadrantes ou apenas em alguns?
  - **Q1**: openSMILE 0.649 → Fusão 0.761 (Δ +0.112)
  - **Q2**: openSMILE 0.405 → Fusão 0.360 (Δ -0.045)
  - **Q3**: openSMILE 0.506 → Fusão 0.600 (Δ +0.093)
  - **Q4**: openSMILE 0.312 → Fusão 0.210 (Δ -0.102)

## 3. Qual tipo de fingerprint ajuda mais: ranking geral ou por bandas?
  - **Fingerprint Rank** supera Band isolado: F1 0.4312 vs 0.4042.

## 4. A fusão melhora Macro F1 e Balanced Accuracy?
  - Cenário OS+AllFP: Δ Macro F1 = +0.0134, Δ Balanced Acc = -0.0054
  - Classificação: **melhora em Macro F1**

## 5. As features de fingerprint aparecem entre as mais importantes?
  - openSMILE: 57.5% da importância total
  - Fingerprint Band: 25.8% da importância total
  - Fingerprint Rank: 16.8% da importância total
  - **Sim**: fingerprints somam 42.5% da importância no cenário OS+AllFP.

## 6. A fusão indica complementaridade ou redundância?
  - **Complementaridade parcial**: ganho pequeno mas positivo (Δ Macro F1 = +1.46pp), sugerindo sobreposição parcial com openSMILE.

## 7. O resultado é suficiente para defender que o fingerprint contribui para a classificação emocional?
  - **Sim, com ressalvas**: 5/9 cenários de fusão apresentam ganho positivo.
  - A evidência sugere que o fingerprint agrega informação, mas o ganho não é sempre expressivo.
  - Recomenda-se reportar tanto os casos de melhora quanto os neutros para objetividade acadêmica.

---
_Gerado em: 2026-06-08 06:32 | Validação: GroupKFold(n=5) por song_id_

Síntese exportada: C:\dev\python\TCC Ajustado\Dados\pycaret_fusion_quadrants_validation\fusion_quadrants_interpretation_summary.md
